 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

 2: Load Dataset

In [9]:
import pandas as pd

df = pd.read_csv("raw_car_dataset.csv")
print(df.head(10))


    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022
5  $1,500     211171.0    4.0          0       ??  2003
6  1500.0     222235.0    4.0          2    Rural  2004
7  $1,500     105068.0    5.0          1     City  2002
8  $2,291      90015.0    4.0          0    Rural  2007
9  1500.0     125976.0    2.0          0     City  2002


3: Load & Inspect

In [10]:
print("Head (10 rows)")
display(df.head(10))

print("Shape:")
print(df.shape)

print("\nInfo:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nLocation Counts:")
print(df["Location"].value_counts(dropna=False))

Head (10 rows)


,Price,Odometer_km,Doors,Accidents,Location,Year
0,"$1,500",137830.0,4.0,1,City,1998
1,4171.0,NaN,4.0,0,Rural,2016
2,"$5,331",107302.0,4.0,0,Suburb,2014
3,1500.0,141838.0,4.0,1,Suburb,1999
4,1500.0,NaN,3.0,0,City,2022
5,"$1,500",211171.0,4.0,0,??,2003
6,1500.0,222235.0,4.0,2,Rural,2004
7,"$1,500",105068.0,5.0,1,City,2002
8,"$2,291",90015.0,4.0,0,Rural,2007
9,1500.0,125976.0,2.0,0,City,2002


Shape:
(145, 6)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    str    
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    str    
 5   Year         145 non-null    int64  
dtypes: float64(2), int64(2), str(2)
memory usage: 6.9 KB

Missing Values:
Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64

Location Counts:
Location
City      59
Suburb    45
Rural     21
Subrb      8
??         7
NaN        5
Name: count, dtype: int64


4: Clean Price

In [11]:
df["Price"] = (
    df["Price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

df["Price"] = pd.to_numeric(df["Price"])

print(df["Price"].dtype)
print("Skewness:", df["Price"].skew())

float64
Skewness: 7.871113660850301


5: Fix Location Errors

In [12]:
df["Location"] = df["Location"].replace({
    "Subrb": "Suburb",
    "??": np.nan
})

print(df["Location"].value_counts(dropna=False))

Location
City      59
Suburb    53
Rural     21
NaN       12
Name: count, dtype: int64


6: Impute Missing Values

In [13]:
df["Odometer_km"] = df["Odometer_km"].fillna(
    df["Odometer_km"].median()
)

df["Doors"] = df["Doors"].fillna(
    df["Doors"].mode()[0]
)

df["Accidents"] = df["Accidents"].fillna(
    df["Accidents"].mode()[0]
)

df["Location"] = df["Location"].fillna(
    df["Location"].mode()[0]
)

print(df.isnull().sum())

Price          0
Odometer_km    0
Doors          0
Accidents      0
Location       0
Year           0
dtype: int64


7: Remove Duplicates

In [14]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Before:", before)
print("After:", after)
print("Removed:", before - after)

Before: 145
After: 140
Removed: 5


8. Outliers (IQR Capping)

In [18]:
print("===== STEP 6: OUTLIER CAPPING =====")

for col in ["Price", "Odometer_km"]:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    print(f"\n{col}")
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)

    # Capping
    df[col] = df[col].clip(lower=lower_bound,
                           upper=upper_bound)

    print(df[col].describe())

===== STEP 6: OUTLIER CAPPING =====

Price
Lower Bound: -2984.625
Upper Bound: 8974.375
count     140.000000
mean     3175.456250
std      2601.848629
min      1500.000000
25%      1500.000000
50%      1500.000000
75%      4489.750000
max      8974.375000
Name: Price, dtype: float64

Odometer_km
Lower Bound: -2.343674706210564
Upper Bound: 2.346272631315266
count    140.000000
mean      -0.027776
std        0.905823
min       -2.147710
25%       -0.584944
50%       -0.068130
75%        0.587542
max        2.346273
Name: Odometer_km, dtype: float64


9: One-Hot Encoding

In [15]:
df = pd.get_dummies(
    df,
    columns=["Location"],
    dtype=int
)

print(df.columns)

Index(['Price', 'Odometer_km', 'Doors', 'Accidents', 'Year', 'Location_City',
       'Location_Rural', 'Location_Suburb'],
      dtype='str')


10: Feature Engineering

In [16]:
CURRENT_YEAR = 2026

df["CarAge"] = CURRENT_YEAR - df["Year"]

df["Km_per_year"] = (
    df["Odometer_km"] /
    df["CarAge"].replace(0, 1)
)

df["Is_Urban"] = (
    df["Location_City"] +
    df["Location_Suburb"]
    > 0
).astype(int)

df["LogPrice"] = np.log1p(df["Price"])

display(
    df[["CarAge", "Km_per_year",
        "Is_Urban", "LogPrice"]].head()
)

,CarAge,Km_per_year,Is_Urban,LogPrice
0,28,4922.500000,1,7.313887
1,10,12854.800000,0,8.336151
2,12,8941.833333,1,8.581482
3,27,5253.259259,1,7.313887
4,4,32137.000000,1,7.313887


11: Scaling

In [17]:
scaler = StandardScaler()

cols_to_scale = [
    "Odometer_km",
    "Doors",
    "Accidents",
    "Year",
    "CarAge",
    "Km_per_year"
]

df[cols_to_scale] = scaler.fit_transform(
    df[cols_to_scale]
)

display(df[cols_to_scale].head())

,Odometer_km,Doors,Accidents,Year,CarAge,Km_per_year
0,0.088107,0.254091,0.316968,-1.686714,1.686714,-0.609073
1,-0.068130,0.254091,-0.820867,0.794617,-0.794617,0.054986
2,-0.425746,0.254091,-0.820867,0.518913,-0.518913,-0.272591
3,0.155570,0.254091,0.316968,-1.548862,1.548862,-0.581383
4,-0.068130,-0.931668,-0.820867,1.621727,-1.621727,1.669210


Save Dataset

In [19]:
df.to_csv(
    "car_l3_clean_ready.csv",
    index=False
)

print("Saved Successfully!")

Saved Successfully!
